In [0]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 10_000


# =========================================================
# 1. PERFECT DATA
#
# House price is determined exactly by:
#
# price = 50 + 4*area + 20*bedrooms - 2*age - 5*distance
#
# No noise.
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
)

df_perfect = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 2. REALISTIC DATA + NOISE
#
# Same underlying relationship,
# but real-world randomness is added.
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 80, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_noise = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 3. NO LINEAR RELATIONSHIP
#
# None of the features actually determines price.
# Price is essentially random.
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = np.random.normal(500, 100, n)

df_no_relationship = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 4. OUTLIERS
#
# Normally realistic data...
# but a few houses have absurd prices.
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 50, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

# Extremely unusual houses
outlier_idx = np.random.choice(n, 20, replace=False)

price[outlier_idx] *= 5

df_outliers = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 5. MULTICOLLINEARITY
#
# Larger houses tend to have more bedrooms.
# Therefore area and bedrooms contain overlapping
# information.
# =========================================================

area = np.random.uniform(40, 300, n)

# Bedrooms strongly related to area
bedrooms_continuous = area / 55 + np.random.normal(0, 0.15, n)

bedrooms = np.clip(
    np.round(bedrooms_continuous),
    1,
    6
)

age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 50, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_multicollinear = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 6. IRRELEVANT FEATURES
#
# Only area actually matters.
# The other variables are noise.
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = (
    100
    + 5 * area
    + np.random.normal(0, 50, n)
)

df_irrelevant = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})


# =========================================================
# 7. NON-LINEAR RELATIONSHIP
#
# Price increases with area,
# but not linearly.
#
# price ≈ area²
# =========================================================

area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 5000, n)

price = (
    2 * area**2
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_nonlinear = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})

In [0]:
#%pip install statsmodels

In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# DF perfect

In [0]:
print(df_perfect.columns)
print(df_perfect.sample(4))

In [0]:
# df_perfect
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_perfect.drop(columns=['price_millions']), df_perfect['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipeline.fit(X_train,y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('====='*10)
print('''area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
)

df_perfect = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})''')
print('====='*10)
print(model.summary(
    yname='price_millions',
    xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']
))

In [0]:
import matplotlib.pyplot as plt

y_pred = model.predict(X_const)
residuals = y_train - y_pred

# Plot Residuals vs Fitted values
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values (Predicted Price)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residuals vs Fitted Plot')
plt.show()

# DF + Noise

In [0]:
print(df_noise.columns)
print(df_noise.sample(3))

In [0]:
# df_noise
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_noise.drop(columns=['price_millions']), df_noise['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 80, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_noise = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

# DF NO LINEAR RELATIONSHIP

In [0]:
print(df_no_relationship.columns)
print(df_no_relationship.sample(3))

In [0]:
# df_no_relationship
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_no_relationship.drop(columns=['price_millions']), df_no_relationship['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = np.random.normal(500, 100, n)

df_no_relationship = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

# DF Outliers

In [0]:
print(df_outliers.columns)
print(df_outliers.sample(3))

In [0]:
# df_outliers
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_outliers.drop(columns=['price_millions']), df_outliers['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 50, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

# Extremely unusual houses
outlier_idx = np.random.choice(n, 20, replace=False)

price[outlier_idx] *= 5

df_outliers = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

# DF multicollinearity

In [0]:
print(df_multicollinear.columns)
print(df_multicollinear.sample(3))

In [0]:
df_multicollinear.corr()

In [0]:
# df_multicollinear
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_multicollinear.drop(columns=['price_millions']), df_multicollinear['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)

# Bedrooms strongly related to area
bedrooms_continuous = area / 55 + np.random.normal(0, 0.15, n)

bedrooms = np.clip(
    np.round(bedrooms_continuous),
    1,
    6
)

age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 50, n)

price = (
    50
    + 4 * area
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_multicollinear = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

In [0]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X_train)
vif_data = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print(vif_data)

## i.e multicollinearity

In [0]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000

# Feature 1
x1 = np.random.normal(100, 20, n)

# Feature 2: altamente relacionada con x1
x2 = x1 * 2 + np.random.normal(0, 2, n)

# Feature 3: prácticamente independiente
x3 = np.random.normal(50, 10, n)

# Target
y = 10 + 3*x1 + 2*x3 + np.random.normal(0, 10, n)

df = pd.DataFrame({
    "target": y,
    "feature_high": x1,
    "feature_high_2": x2,
    "feature_null": x3
})

print(df.head())

|        VIF | Interpretación                               |
| ---------: | -------------------------------------------- |
|      **1** | 🟢 Sin multicolinealidad                     |
|    **1–5** | 🟢 Baja / generalmente aceptable             |
|   **5–10** | 🟡 Multicolinealidad moderada/alta → revisar |
|    **>10** | 🔴 Multicolinealidad alta                    |
| **>20–30** | 🔴🔴 Muy alta                                |
|      **∞** | 🔴 Colinealidad perfecta                     |


In [0]:
X = df.drop(columns="target")

X_vif = sm.add_constant(X)
vif_data = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print(vif_data)

# DF irrelevant features

In [0]:
print(df_irrelevant.columns)
print(df_irrelevant.sample(3))

In [0]:
# df_irrelevant
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_irrelevant.drop(columns=['price_millions']), df_irrelevant['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

price = (
    100
    + 5 * area
    + np.random.normal(0, 50, n)
)

df_irrelevant = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

### Feature Selection & Interpretation

Based on the coefficient weights and pvalues:

* **`area_m2`** is the only feature driving the target variable (`price_millions`), showing high statistical significance (pvalue < 0.001$).
* **`bedrooms`**, **`age_years`**, and **`distance_center_km`** do not contribute meaningfully to the model's predictive power (pvalues $> 0.05$ or negligible effect sizes).

**Action:** Remove `bedrooms`, `age_years`, and `distance_center_km` to keep the model parsimonious and prevent fitting on random noise.

# DF no linear relationship

In [0]:
print(df_nonlinear.columns)
print(df_nonlinear.sample(3))

In [0]:
# df_nonlinear
cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km']

X, y = df_nonlinear.drop(columns=['price_millions']), df_nonlinear['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(''' 
area = np.random.uniform(40, 300, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 40, n)
distance = np.random.uniform(1, 30, n)

noise = np.random.normal(0, 5000, n)

price = (
    2 * area**2
    + 20 * bedrooms
    - 2 * age
    - 5 * distance
    + noise
)

df_nonlinear = pd.DataFrame({
    "area_m2": area,
    "bedrooms": bedrooms,
    "age_years": age,
    "distance_center_km": distance,
    "price_millions": price
})
      ''')
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km']))

In [0]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

df_nonlinear

X_linear = sm.add_constant(df_nonlinear[["area_m2"]])
model_linear = sm.OLS(df_nonlinear["price_millions"], X_linear).fit()
y_pred_linear = model_linear.predict(X_linear)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- GRÁFICO 1: Ajuste de la recta sobre los datos reales ---
# Muestra aleatoria de 500 puntos para no saturar el gráfico
idx = np.random.choice(n, 500, replace=False)

sns.scatterplot(
    x=df_nonlinear["area_m2"].iloc[idx], 
    y=df_nonlinear["price_millions"].iloc[idx], 
    alpha=0.4, color="gray", label="Datos reales (con ruido)", ax=axes[0]
)

# la recta que trazó OLS
sort_idx = np.argsort(df_nonlinear["area_m2"])
axes[0].plot(
    df_nonlinear["area_m2"].iloc[sort_idx], 
    y_pred_linear.iloc[sort_idx], 
    color="red", linewidth=2.5, label=f"Recta OLS (R² = {model_linear.rsquared:.3f})"
)

axes[0].set_title("1. Recta OLS intentando ajustar una curva ($Y = 2 \cdot area^2$)", fontsize=12)
axes[0].set_xlabel("Área ($m^2$)")
axes[0].set_ylabel("Precio (millones)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- GRÁFICO 2: Análisis de Residuos (El patrón en U) ---
residuals = model_linear.resid

sns.scatterplot(
    x=df_nonlinear["area_m2"].iloc[idx], 
    y=residuals.iloc[idx], 
    alpha=0.4, color="purple", ax=axes[1]
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1.5)

# Línea de tendencia de los residuos (LOWESS)
sns.regplot(
    x=df_nonlinear["area_m2"], 
    y=residuals, 
    scatter=False, lowess=True, color="red", ax=axes[1], line_kws={"linewidth": 2.5}
)

axes[1].set_title("2. Residuos vs. Área (Delata la falta de linealidad)", fontsize=12)
axes[1].set_xlabel("Área ($m^2$)")
axes[1].set_ylabel("Residuo (Real - Predicho)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
# Código para verificar dispersión de residuos
plt.scatter(y_pred_linear, residuals, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Valores Predichos")
plt.ylabel("Residuos")
plt.show()

In [0]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(df_nonlinear["price_millions"], y_pred_linear))
mae = mean_absolute_error(df_nonlinear["price_millions"], y_pred_linear)

print(f"Error promedio (MAE): {mae:.2f} ")
print(f"RMSE: {rmse:.2f} ")

In [0]:
import scipy.stats as stats

# Q-Q Plot
stats.probplot(residuals, dist="norm", plot=plt)
plt.show()

## Initial Diagnostic & Model Failure Summary

* **Context & Setup:** A baseline linear regression (OLS) model was fitted to predict property prices (`price_millions`) using linear features.
* **The Illusion ($R^2 = 95.4\%$):** The model yielded a high $R^2$ of $0.954$, giving a false impression of strong predictive performance simply because the overall trend was steeply increasing compared to a flat baseline mean.
* **The Real-World Failure:** 
  * Absolute error metrics revealed massive real-world inaccuracies, with a **MAE of 9,360 million** and a **RMSE of 11,240 million**.
  * The model produced nonsensical outputs (e.g., negative price predictions for smaller property areas around 40m^2).
* **Root Cause (Residual Analysis):** 
  * The **Residuals vs. Feature / Fitted Values** plot exhibited a pronounced **"U" shape (parabola)**, violating the fundamental OLS assumption of linearity.
  * The **Q-Q Plot** displayed an "S" curve deviation, confirming that the residual errors were non-normally distributed due to omitted non-linear terms ($area^2$).

In [0]:
df_nonlinear["area_m2_sq"] = df_nonlinear["area_m2"] ** 2

cols_to_scale = ['area_m2','bedrooms','age_years','distance_center_km','area_m2_sq']

X, y = df_nonlinear.drop(columns=['price_millions']), df_nonlinear['price_millions']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ('scaler',StandardScaler()),
    ('regressor',LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

In [0]:
score = r2_score(y_test, pipeline.predict(X_test))

X_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_const).fit()

print(score)
print('===='*10)
print(model.summary(yname='price_millions', xname=['intercept','area_m2','bedrooms','age_years','distance_center_km','area_m2_sq']))

In [0]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

df_nonlinear

X_linear = sm.add_constant(df_nonlinear[["area_m2_sq"]])
model_linear = sm.OLS(df_nonlinear["price_millions"], X_linear).fit()
y_pred_linear = model_linear.predict(X_linear)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- GRÁFICO 1: Ajuste de la recta sobre los datos reales ---
# Muestra aleatoria de 500 puntos para no saturar el gráfico
idx = np.random.choice(n, 500, replace=False)

sns.scatterplot(
    x=df_nonlinear["area_m2_sq"].iloc[idx], 
    y=df_nonlinear["price_millions"].iloc[idx], 
    alpha=0.4, color="gray", label="Datos reales (con ruido)", ax=axes[0]
)

# Recta que trazó OLS
sort_idx = np.argsort(df_nonlinear["area_m2_sq"])
axes[0].plot(
    df_nonlinear["area_m2_sq"].iloc[sort_idx], 
    y_pred_linear.iloc[sort_idx], 
    color="red", linewidth=2.5, label=f"Recta OLS (R² = {model_linear.rsquared:.3f})"
)

axes[0].set_title("1. Recta OLS intentando ajustar una curva ($Y = 2 \cdot area^2$)", fontsize=12)
axes[0].set_xlabel("Área ($m^2$)")
axes[0].set_ylabel("Precio (millones)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- GRÁFICO 2: Análisis de Residuos ---
residuals = model_linear.resid

sns.scatterplot(
    x=df_nonlinear["area_m2_sq"].iloc[idx], 
    y=residuals.iloc[idx], 
    alpha=0.4, color="purple", ax=axes[1]
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1.5)

# Línea de tendencia de los residuos (LOWESS)
sns.regplot(
    x=df_nonlinear["area_m2_sq"], 
    y=residuals, 
    scatter=False, lowess=True, color="red", ax=axes[1], line_kws={"linewidth": 2.5}
)

axes[1].set_title("2. Residuos vs. Área (Delata la falta de linealidad)", fontsize=12)
axes[1].set_xlabel("Área ($m^2$)")
axes[1].set_ylabel("Residuo (Real - Predicho)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
# Código para verificar dispersión de residuos
plt.scatter(y_pred_linear, residuals, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Valores Predichos")
plt.ylabel("Residuos")
plt.show()

In [0]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(df_nonlinear["price_millions"], y_pred_linear))
mae = mean_absolute_error(df_nonlinear["price_millions"], y_pred_linear)

print(f"Error promedio (MAE): {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [0]:
import scipy.stats as stats

# Q-Q Plot
stats.probplot(residuals, dist="norm", plot=plt)
plt.show()

In [0]:
X_multi = sm.add_constant(df_nonlinear[["area_m2", "bedrooms", "age_years", "distance_center_km","area_m2_sq"]])
model_multi = sm.OLS(df_nonlinear["price_millions"], X_multi).fit()

# Graficar residuos vs Valores Predichos (y_hat)
plt.scatter(model_multi.fittedvalues, model_multi.resid, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Valores Predichos (y_hat)")
plt.ylabel("Residuos")
plt.title("Residuos del Modelo Multivariable Completo")
plt.show()

## 2. Polynomial Feature Implementation & Resolution Summary

* **Feature Engineering:** A quadratic feature `area_m2_sq` (area^2) was created and added to the feature matrix to allow the linear regression equation to capture parabolic curvature ($\hat{Y} = \beta_0 + \beta_1 \cdot area + \beta_2 \cdot area^2).
* **Metric Collapse:**
  * **MAE dropped drastically** from $9,360$ million to **3,967 million**.
  * **RMSE dropped** from $11,240$ million to **4,974 million**.
  * **$R^2$ rose** from $0.954$ to **0.991**.
* **Theoretical Limit Reached:** The final RMSE of $\sim 4,974$ matches the irreducible random Gaussian noise ($\sigma = 5,000$) injected into the synthetic dataset, proving the model extracted $100\%$ of the underlying mathematical pattern.
* **Residual Verification:**
  * The parabolic "U" pattern completely disappeared; residuals now form a uniform, random cloud centered at $0$ (true white noise).
  * The Q-Q plot points aligned almost perfectly along the diagonal, verifying error normality.
  * The model correctly recovered the true coefficient $\beta_2 \approx 1.995$ (matching the generative baseline $2 \cdot area^2$).
* **Parsimony Next Step:** Non-significant features (`bedrooms`, `age_years`, `distance_center_km`) with $p > 0.05$ can now be safely dropped, leaving a clean, highly accurate model using only `intercept` and `area_m2_sq`.